# L14a: Dominant Eigenpairs and Power Iteration

A matrix can rotate and stretch most vectors, but an eigenvector keeps its direction. Its eigenvalue records the associated amplification. Those two objects let us predict convergence, long-run behavior, and stability without following every intermediate step.

> **Learning objectives**
>
> - Interpret an eigenpair as an invariant direction and its amplification factor.
> - Implement power iteration to estimate a dominant eigenpair.
> - Validate an estimated eigenpair using a residual and a library result.
> - Connect spectral information to iterative solvers, Markov models, and explicit-Euler stability.


## Setup

Run the local setup cell first. It activates the pinned course environment, loads `LinearAlgebra`, and includes [`../src/Week14Core.jl`](../src/Week14Core.jl).


In [ ]:
include(joinpath(@__DIR__, "Include.jl"))

## An eigenpair is a predictive shortcut

For a square matrix `A`, an eigenvector `v` and eigenvalue `lambda` satisfy `A * v = lambda * v`. Repeated application therefore gives `A^k * v = lambda^k * v`. The direction is unchanged; only its magnitude changes.

That observation turns a matrix into a prediction:

- `abs(lambda) < 1`: the mode decays;
- `abs(lambda) == 1`: the mode persists;
- `abs(lambda) > 1`: the mode grows.

The dominant eigenvalue is the eigenvalue with largest magnitude. Unless the initial vector misses its eigenvector exactly, repeated matrix-vector multiplication increasingly emphasizes that dominant direction.


In [ ]:
A = [
    4.0  1.0  0.0
    1.0  3.0  1.0
    0.0  1.0  2.0
]
reference = eigen(Symmetric(A))

## Compute the dominant mode with power iteration

Power iteration repeats three operations: multiply by `A`, normalize the result, and estimate the eigenvalue with the Rayleigh quotient. The stopping test should measure the eigenpair claim itself, not merely whether two successive vectors look similar. Here we stop when `norm(A * v - lambda * v)` is below the requested tolerance.


In [ ]:
result = power_iteration(A, ones(3); tolerance = 1e-10)

## Validate the computational claim

A plausible eigenvalue is not enough. We check the residual, normalization, and agreement with Julia's maintained eigensolver. Eigenvectors are only defined up to sign, so compare their directions through the absolute dot product.


In [ ]:
dominant_index = argmax(reference.values)
reference_value = reference.values[dominant_index]
reference_vector = reference.vectors[:, dominant_index]

@test result.converged
@test result.residual_norm <= 1e-10
@test isapprox(result.value, reference_value; atol = 1e-9)
@test isapprox(abs(dot(result.vector, reference_vector)), 1.0; atol = 1e-8)

## The spectral gap controls convergence speed

Power iteration converges quickly when the dominant magnitude is well separated from the second-largest magnitude. It converges slowly when those magnitudes are close. This is an algorithmic consequence of the ratio `abs(lambda_2 / lambda_1)`, not a performance accident.


In [ ]:
fast = power_iteration(Diagonal([5.0, 1.0, 0.5]), ones(3); tolerance = 1e-9)
slow = power_iteration(Diagonal([5.0, 4.9, 0.5]), ones(3); tolerance = 1e-9)
(fast_iterations = fast.iterations, slow_iterations = slow.iterations)

## One spectral idea, three course connections

| Course setting | Matrix applied repeatedly | What eigenvalues predict |
|---|---|---|
| Stationary iterative solvers | Iteration matrix | Whether the error decays |
| Markov models | Transition matrix | Long-run modes and persistence |
| Explicit Euler | Update matrix | Whether a time step is numerically stable |

This is why eigenvalues belong in the Fall prerequisite chain: they organize computations students have already used and prepare the stability analysis in Week 15.


### Long-run behavior of a Markov model

For the column-stochastic transition matrix below, the eigenvalue `1` corresponds to a stationary distribution. Power iteration recovers that distribution by repeatedly propagating an initial state.


In [ ]:
transition = [0.85 0.25; 0.15 0.75]
stationary_mode = power_iteration(transition, [0.5, 0.5]; tolerance = 1e-12)
stationary_probability = stationary_mode.vector ./ sum(stationary_mode.vector)

@test isapprox(transition * stationary_probability, stationary_probability; atol = 1e-11)
stationary_probability

### Bridge to Week 15: explicit-Euler stability

For the scalar mode `du/dt = lambda * u`, explicit Euler updates the state by the amplification factor `1 + step * lambda`. The numerical mode is stable when the magnitude of that factor is at most one. A stable physical mode can therefore become an unstable numerical calculation when the step is too large.


In [ ]:
lambda = -2.0
steps = [0.25, 1.0, 1.1]
[(step = step, amplification = 1 + step * lambda, stable = explicit_euler_stable(lambda, step)) for step in steps]

## Boundary with CHEME 5820

Fall stops after the dominant eigenpair, power iteration, residual validation, and spectral interpretation. CHEME 5820 extends this prerequisite to full eigendecomposition, QR iteration, classical and modified Gram–Schmidt, covariance eigendecomposition, and principal component analysis.

That division makes this lecture useful preparation rather than a duplicate of the Spring unit.


## Summary

- Eigenvectors are invariant directions; eigenvalues are their amplification factors.
- Power iteration estimates the dominant eigenpair through repeated matrix-vector multiplication.
- The eigenpair residual makes the numerical claim testable.
- Spectral magnitude predicts convergence, long-run behavior, and explicit-Euler stability.

**Exit question:** In one of the three course settings above, identify the repeatedly applied matrix and state what its dominant eigenvalue tells you.
